# 03 — CMF Model Search (AADT as Primary Term)

A **Crash Modification Factor (CMF)** model separates the AADT (traffic volume)
effect from site-specific characteristics.

**Two-component model structure:**
```
log(mu_i) = [alpha_0 + SUM_k alpha_k * X_ki]             (Baseline block — Component A)
          + [beta_0  + SUM_k beta_k  * X_ki] * log(AADT_i)  (Local block — Component B)
```

- **Component A** (baseline): independent of traffic volume
- **Component B** (local): each coefficient is scaled by log(AADT)

**Reading CMF values:**
```
alpha_k = 0.15  →  CMF_A = exp(0.15) = 1.16  (+16% more crashes)
alpha_k = -0.30 →  CMF_A = exp(-0.30) = 0.74  (-26% fewer crashes)

beta_k at mean AADT = 10,000:
    CMF_B = AADT_mean ^ beta_k = 10000 ^ 0.05 ≈ 1.26  (+26%)
```

**Two routes shown in this notebook:**

| Route | When to use |
| --- | --- |
| **A. JAX full-flexibility search** | Full role system (0–8), latent classes, ZI, random params |
| **B. Classic GA search** | Original two-component AADT structure; fast; no LC |

**Previous:** [02_latent_class_fc_validation.ipynb](02_latent_class_fc_validation.ipynb)
**Next:** [04_linear_speed_prediction.ipynb](04_linear_speed_prediction.ipynb)

In [ ]:
import numpy as np
import pandas as pd

from metacountregressor import (
    CMFExperimentBuilder,
    ModelConstraints,
    SearchOutputConfig,
    load_example16_3_model_data,
    load_book_cmf_spec,
    describe_book_cmf_spec,
    get_help,
)

# Uncomment to read the full CMF workflow guide:
# get_help('cmf')

print('Imports OK')

## 1 — Load data

In [ ]:
df = load_example16_3_model_data()

print(f'Rows: {len(df)}  |  Columns: {df.shape[1]}')
print(f'AADT range: {df["AADT"].min():.0f} – {df["AADT"].max():.0f} vehicles/day')
print()
print('FREQ (crash count) summary:')
print(df['FREQ'].describe().round(2))
print()
print('log(AADT) range:', np.log(df['AADT']).min().round(2), '–', np.log(df['AADT']).max().round(2))

In [ ]:
df[['ID', 'FREQ', 'AADT', 'LENGTH', 'URB', 'ACCESS', 'CURVES', 'OFFSET']].head(6)

## 2 — Define baseline and local variable sets

- **Baseline variables** → Component A (affect crashes independently of AADT)
- **Local variables** → Component B (their effect is multiplied by `log(AADT)`)

A variable can appear in both blocks.

In [ ]:
# ── What to change here ────────────────────────────────────────────────────────
#
#   baseline_vars : variables that affect crash risk independently of traffic volume
#                   (e.g. urban/rural indicator, access control type, grade break)
#
#   local_vars    : variables whose effect on crash risk changes with traffic volume
#                   (e.g. horizontal curves, lane width)
#
#   aadt_col      : column name for AADT (must be positive for log transform)

baseline_vars = ['URB', 'ACCESS', 'GRADEBR', 'CURVES']   # Component A
local_vars    = ['CURVES', 'WIDTH']                        # Component B

cmf = CMFExperimentBuilder(
    df=df,
    y_col='FREQ',
    aadt_col='AADT',
    baseline_vars=baseline_vars,
    local_vars=local_vars,
)

print('CMFExperimentBuilder created.')
print(f'Baseline block variables : {baseline_vars}')
print(f'Local block variables    : {local_vars}')

## Route A — Full-flexibility JAX search

`build_jax_count_evaluator()` applies the full `ExperimentBuilder` role system
to the CMF structure.  `log(AADT)` is forced fixed (role 1); all other variables
can take roles 0–8 subject to constraints.

Use this route when you want:
- Random CMF parameters
- Latent classes with FC-driven membership
- Zero-inflation
- Full BIC-based structure search

In [ ]:
c = (
    ModelConstraints()
    # Road geometry: not plausible as ZI terms
    .no_zi('CURVES', 'WIDTH', 'GRADEBR', 'ACCESS', 'URB')
    # Allow random curvature effect (lognormal = positive direction)
    .allow_random('CURVES', distributions=['lognormal'])
    # Binary dummies: no individual taste variation
    .no_random('URB', 'GRADEBR')
    # To add latent-class membership, uncomment:
    # .membership_only('FC_ENCODED')
)

print('CMF search constraints:')
print(c)

In [ ]:
# Build the JAX CMF evaluator
builder_jax, evaluator_jax, meta = cmf.build_jax_count_evaluator(
    id_col='ID',
    offset_col='OFFSET',
    constraints=c,
    max_latent_classes=1,   # set to 2 to add latent-class capability
    R=200,
)

print('JAX CMF evaluator ready.')
print('Search variables :', meta['search_vars'])
print('log(AADT) column :', meta['log_aadt_col'])

In [ ]:
print('Running JAX CMF search (max_iter=50 — demo) ...')
print()

result_jax = builder_jax.run(
    evaluator=evaluator_jax,
    algo='sa',
    max_iter=50,
    seed=42,
    output_config=SearchOutputConfig(
        output_dir='results',
        experiment_name='cmf_jax_search',
        search_description='CMF JAX full-flex search on Example 16-3',
    ),
)

print('─' * 60)
print(f'  Algorithm      : {result_jax.algorithm}')
print(f'  Best BIC       : {result_jax.best_score:.3f}')
print(f'  Iterations     : {result_jax.iteration}')
print(f'  Time           : {result_jax.elapsed_time:.1f} s')
print('─' * 60)
print()
print('Best CMF structure found:')
for k, v in result_jax.best_spec.items():
    if v:
        print(f'  {k:<20}: {v}')

## Route B — Classic GA CMF search

The classic GA search directly optimises the two-component AADT structure.
It is faster and produces immediately interpretable CMF values, but does not
support latent classes.

In [ ]:
print('Running classic GA CMF search ...')

search_result = cmf.run_search(R=200)

print()
print('─' * 60)
print('GA search complete.')
print(f'Selected baseline vars : {search_result.selected_baseline}')
print(f'Selected local vars    : {search_result.selected_local}')
print('─' * 60)

In [ ]:
# Re-fit and compute standard errors with more draws
print('Fitting selected CMF structure with R=500 draws ...')

fit_result = cmf.fit_best_model(search_result, final_R=500)

print()
print('─' * 60)
print('CMF model fit results:')
print('─' * 60)
cmf.print_report(search_result, fit_result)

## 3 — Reading CMF values from the output

Each estimated coefficient translates directly to a crash modification factor:

In [ ]:
# CMF interpretation helper
print('CMF INTERPRETATION GUIDE')
print('=' * 50)
print()
print('Component A coefficients (baseline block):')
print('  alpha_k = 0.15  →  CMF = exp(0.15) = {:.3f}  (+{:.0%} crashes)'.format(
    np.exp(0.15), np.exp(0.15) - 1))
print('  alpha_k = -0.30 →  CMF = exp(-0.30) = {:.3f}  ({:.0%} fewer crashes)'.format(
    np.exp(-0.30), np.exp(-0.30) - 1))
print()

# Component B at example AADT values
example_aadts = [5000, 10000, 25000, 50000]
example_beta  = 0.05
print('Component B coefficient (local block):')
print(f'  beta_k = {example_beta}  →  CMF = AADT^beta_k')
print()
print(f'  {"AADT":>8}  {"CMF":>8}')
for aadt in example_aadts:
    cmf_val = aadt ** example_beta
    print(f'  {aadt:>8,}  {cmf_val:>8.3f}')

## 4 — Fit the reference CMF specification

The package includes a reference CMF spec.  Use it to compare against the searched model.

In [ ]:
# Describe the reference spec
describe_book_cmf_spec()

In [ ]:
ref_spec = load_book_cmf_spec()

# Build a manual CMF spec from the reference
manual_spec = cmf.make_manual_cmf_spec(
    baseline_fixed=ref_spec['baseline_fixed'],
    baseline_random=ref_spec['baseline_random'],
    local_fixed=ref_spec['local_fixed'],
    local_random=ref_spec['local_random'],
    hetro_in_means=ref_spec['hetro_in_means'],
    membership_terms=ref_spec['membership_terms'],
    dispersion=ref_spec['dispersion'],
    latent_classes=ref_spec['latent_classes'],
)

print('Reference manual spec:')
for k, v in manual_spec.items():
    print(f'  {k:<20}: {v}')
print()

ref_fit = cmf.fit_manual_cmf_model(
    id_col='ID',
    offset_col='OFFSET',
    manual_spec=manual_spec,
    model='nb',
    R=300,
)

print('─' * 60)
print('Reference CMF model fit:')
print('─' * 60)
print(ref_fit)

## 5 — Extend to latent-class CMF

To allow the CMF coefficients to vary across latent sub-populations,
set `max_latent_classes=2` and add a membership variable.

In [ ]:
# Uncomment to run a 2-class LC CMF search:

# c_lc = (
#     ModelConstraints()
#     .no_zi('CURVES', 'WIDTH', 'GRADEBR')
#     .no_random('URB', 'GRADEBR')
#     .allow_random('CURVES', distributions=['lognormal'])
#     .membership_only('FC_ENCODED')  # FC drives class membership only
# )
#
# builder_lc, evaluator_lc, meta_lc = cmf.build_jax_count_evaluator(
#     id_col='ID',
#     offset_col='OFFSET',
#     constraints=c_lc,
#     max_latent_classes=2,   # ← enable latent classes
#     R=150,
# )
#
# result_lc = builder_lc.run(evaluator_lc, algo='sa', max_iter=100, seed=7)
# print('2-class CMF BIC:', result_lc.best_score)
# print('Compared to 1-class JAX BIC:', result_jax.best_score)

print('2-class CMF search commented out — uncomment the block above to run it.')
print('See get_help("cmf") and get_help("latent_class") for more details.')

## Tuning guide

| Setting | Value | Effect |
| --- | --- | --- |
| `baseline_vars` | Add variables | Wider Component A search |
| `local_vars` | Add variables | Wider Component B search |
| `max_latent_classes=2` | plus `membership_only('FC_ENCODED')` | Latent-class CMF |
| `default_roles` include `6` | Enables zero inflation | Models structural zeros |
| `R` | 200 → 500 → 1000 | More stable, slower |
| `max_iter` | 50 → 1000 → 5000 | Better search, more time |

```python
get_help('cmf')          # full CMF workflow
get_help('latent_class') # latent class guide
get_help('constraints')  # constraint API
```

**Next:** [04_linear_speed_prediction.ipynb](04_linear_speed_prediction.ipynb)